In [4]:
import pandas as pd
import numpy as np
import os

# Check if data folder exists, if not create it
if not os.path.exists('data'):
    os.makedirs('data')

# 1. Synthetic Data Generate karna (1000 rows)
np.random.seed(42)
data = {
    'room_id': np.random.randint(1, 50, 1000),
    'time_of_day': np.random.choice(['Morning', 'Afternoon', 'Evening', 'Night'], 1000),
    'is_emergency': np.random.choice([0, 1], 1000, p=[0.8, 0.2]), 
    'sensor_capacity': np.random.randint(10, 100, 1000),
    'actual_crowd': np.random.randint(0, 120, 1000)
}

df = pd.DataFrame(data)

# 2. Target Variable banana (Crowd Density: 0=Low, 1=Medium, 2=High)
conditions = [
    (df['actual_crowd'] < 30),
    (df['actual_crowd'] >= 30) & (df['actual_crowd'] < 70),
    (df['actual_crowd'] >= 70)
]
choices = ['Low', 'Medium', 'High']
df['crowd_density'] = np.select(conditions, choices, default='Unknown')

# 3. Data ko CSV file mein save karna
df.to_csv('data/crowd_data.csv', index=False)
print("✅ Dataset successfully generated and saved to data/crowd_data.csv")
print(df.head())

✅ Dataset successfully generated and saved to data/crowd_data.csv
   room_id time_of_day  is_emergency  sensor_capacity  actual_crowd  \
0       39     Morning             0               10            14   
1       29       Night             1               81            98   
2       15       Night             0               96            36   
3       43     Evening             0               72             0   
4        8   Afternoon             0               76            66   

  crowd_density  
0           Low  
1          High  
2        Medium  
3           Low  
4        Medium  


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Data Load karo
df = pd.read_csv('data/crowd_data.csv')

# 2. Text ko Numbers mein badlo (Preprocessing)
le_time = LabelEncoder()
le_density = LabelEncoder()

df['time_num'] = le_time.fit_transform(df['time_of_day'])
df['density_num'] = le_density.fit_transform(df['crowd_density'])

# 3. Features (X) aur Target (y) alag karo
# Hum room_id, time, emergency status aur capacity dekh kar density predict karenge
X = df[['room_id', 'time_num', 'is_emergency', 'sensor_capacity']]
y = df['density_num']

# 4. Data ko Training aur Testing mein baanto (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Model Train karo (Random Forest Algorithm)
model = RandomForestClassifier()
model.fit(X_train, y_train)

# 6. Accuracy check karo
accuracy = model.score(X_test, y_test)
print(f"✅ Model Training Complete! Accuracy: {accuracy * 100:.2f}%")

# 7. Model ko file mein save karo taaki baad mein use ho sake
joblib.dump(model, 'data/safeflow_model.pkl')
joblib.dump(le_density, 'data/label_encoder.pkl')
print("💾 Model successfully saved to data/safeflow_model.pkl")

✅ Model Training Complete! Accuracy: 31.50%
💾 Model successfully saved to data/safeflow_model.pkl


In [6]:
   import joblib
   import numpy as np

   # 1. Saved Model aur Label Encoder ko load karo
   model = joblib.load('data/safeflow_model.pkl')
   le_density = joblib.load('data/label_encoder.pkl')

   # 2. Ek hypothetical situation banao (Test Input)
   # Format: [room_id, time_num, is_emergency, sensor_capacity]
   # Note: time_num (0=Afternoon, 1=Evening, 2=Morning, 3=Night)
   new_data = np.array([[10, 1, 0, 50]]) 

   # 3. Model se prediction lo
   prediction_num = model.predict(new_data)

   # 4. Number ko wapas text (Low/Medium/High) mein badlo
   prediction_text = le_density.inverse_transform(prediction_num)

   print(f"🔮 Model Prediction for Room 10: {prediction_text[0]}")

🔮 Model Prediction for Room 10: Medium


c:\Users\ANSHITA\Documents\GitHub\safeflow-intelligent-emergeny-evacuation-crowd-routing-system\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
